# Streaming Signal Decomposition — Full Framework Demo

This notebook is a **complete, runnable walkthrough** of every feature of the
Streaming SSD framework developed as part of the DACS bachelor thesis
(Maastricht University, Spring 2026). Executing it top-to-bottom verifies:

1. All four synthetic signal generators
2. Both decomposition engines (SSD, SSA) via the `get_engine` factory, the
   `DecompositionEngine` base class, and the `RankOneUpdater` stub
3. All similarity metrics (`d_corr`, `d_freq`, `w_correlation`, `subspace_angle`)
4. The full streaming pipeline (`WindowManager`, `ComponentMatcher`,
   `TrajectoryStore`), including stateless/stateful APIs and all distance modes
5. All stability metrics (`qrf`, `nmse`, `energy_continuity`,
   `singular_value_drift`, `dominant_frequency`, `freq_drift_aggregate`,
   `matching_confidence`)
6. Every visualization function in `src/visualization/`
7. Robustness to noise (QRF vs SNR sweep)
8. Dynamic component onset detection
9. Integration with `experiments/run_experiment.py` via YAML configs


## 0. Setup — imports, seeds, output directory

In [76]:
import sys, os, json, yaml, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

# ---- signal generators ---------------------------------------------------
from experiments.synthetic.generators import (
    two_sinusoids, chirp_plus_sinusoid, rossler, component_onset,
)

# ---- decomposition engines ----------------------------------------------
from src.engines import (
    DecompositionEngine, get_engine,
    SSD, SSA, RankOneUpdater,
    build_trajectory_matrix, svd_decompose, diagonal_averaging, auto_ssa,
)

# ---- similarity metrics --------------------------------------------------
from src.metrics.similarity import d_corr, d_freq, w_correlation, subspace_angle

# ---- stability metrics ---------------------------------------------------
from src.metrics.stability import (
    qrf, nmse, energy_continuity, singular_value_drift,
    dominant_frequency, freq_drift_aggregate, matching_confidence,
)

# ---- streaming pipeline --------------------------------------------------
from src.streaming.window_manager import WindowManager
from src.streaming.component_matcher import ComponentMatcher
from src.streaming.trajectory_store import TrajectoryStore

# ---- visualization -------------------------------------------------------
from src.visualization import (
    plot_decomposition, plot_trajectory_overlay, plot_component_spectra,
    plot_matching_graph, plot_metrics_over_windows, plot_pipeline_dashboard,
    plot_window_reconstruction, plot_window_grid, plot_nmse_over_time,
)
from src.visualization.plot_metrics import plot_metrics

# ---- experiment runner ---------------------------------------------------
from experiments.run_experiment import build_pipeline, run as run_experiment

FS = 1000.0
SEED = 42
np.random.seed(SEED)

RESULTS_DIR = Path("../results/demo_run")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results will be written to: {RESULTS_DIR.resolve()}")


Results will be written to: /Users/sachaloeb/Documents/StreamingSignalDecompositionRTDA/results/demo_run


## 1. Signal generators

We demonstrate all four synthetic generators. `chirp_plus_sinusoid` is kept as
the primary signal for later sections because it is non-stationary and
therefore exercises every part of the streaming pipeline.

In [77]:
N = 3000

sig_two = two_sinusoids(N=N, f1=20.0, f2=80.0, fs=FS, seed=SEED)
sig_chirp = chirp_plus_sinusoid(N=N, f_sin=50.0, f_start=10.0, f_end=150.0,
                                fs=FS, snr_db=20.0, seed=SEED)
sig_ross = rossler(N=N, dt=0.02, seed=SEED)
sig_onset = component_onset(N=N, f_steady=30.0, f_onset=90.0,
                            onset_sample=N // 2, fs=FS, seed=SEED)

assert sig_two.shape == sig_chirp.shape == sig_ross.shape == sig_onset.shape == (N,)

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=False)
t = np.arange(N) / FS
for ax, sig, name in zip(
    axes,
    [sig_two, sig_chirp, sig_ross, sig_onset],
    ["two_sinusoids", "chirp_plus_sinusoid", "rossler", "component_onset"],
):
    ax.plot(t if name != "rossler" else np.arange(N), sig, linewidth=0.7)
    ax.set_title(name)
    ax.set_ylabel("amp")
axes[-1].set_xlabel("time (s) / samples")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "01_generators.png", dpi=120, bbox_inches="tight")
plt.show()

# primary signal for the rest of the notebook
signal = sig_chirp
print("Primary signal:", signal.shape, "energy =", round(float(np.dot(signal, signal)), 2))


Primary signal: (3000,) energy = 3052.62


## 2. Decomposition engines

### 2.1 The `DecompositionEngine` base class and `get_engine` factory

`get_engine(name, fs, **kwargs)` dispatches by name. Both `SSD` and `SSA`
derive from the abstract `DecompositionEngine` class.

In [78]:
eng_ssd = get_engine("ssd", fs=FS, nmse_threshold=0.01, max_iter=10)
eng_ssa = get_engine("ssa", fs=FS, n_components=3)
assert isinstance(eng_ssd, DecompositionEngine)
assert isinstance(eng_ssa, DecompositionEngine)
print("ssd ->", type(eng_ssd).__name__, "| ssa ->", type(eng_ssa).__name__)
try:
    get_engine("unknown", fs=FS)
except ValueError as exc:
    print("Correctly rejected unknown engine:", exc)


ssd -> SSD | ssa -> SSA
Correctly rejected unknown engine: Unknown engine 'unknown'. Available: ['ssa', 'ssd']


### 2.2 SSD — full offline decomposition

We show the wrapped trajectory matrix, the automatically selected window
length `M = 1.2 * fs / f_max`, and the iterative extraction with the NMSE
stopping criterion.

In [79]:
ssd = SSD(fs=FS, nmse_threshold=0.01, max_iter=8)

# --- automatic window-length selection ---
M_auto = ssd._choose_window_length(signal)
freqs = np.fft.rfftfreq(len(signal), d=1.0 / FS)
f_max = float(freqs[np.argmax(np.abs(np.fft.rfft(signal)) ** 2)])
print(f"Dominant freq f_max = {f_max:.2f} Hz  -> M = 1.2*fs/f_max = {M_auto}")

# --- wrapped trajectory matrix ---
X_wrapped = SSD._build_trajectory_matrix(signal, M_auto)
print("Wrapped trajectory matrix shape:", X_wrapped.shape)
assert X_wrapped.shape == (M_auto, len(signal))

# --- full decomposition ---
ssd_components = ssd.fit(signal)
ssd_residual = ssd_components[-1]
ssd_comps_only = ssd_components[:-1]
recon_ssd = np.sum(ssd_comps_only, axis=0)
print(f"SSD extracted {len(ssd_comps_only)} components "
      f"(NMSE of residual = {nmse(ssd_residual, signal):.5f})")


Dominant freq f_max = 50.00 Hz  -> M = 1.2*fs/f_max = 24
Wrapped trajectory matrix shape: (24, 3000)
SSD extracted 5 components (NMSE of residual = 0.00630)


### 2.3 SSA — `build_trajectory_matrix`, `svd_decompose`, `diagonal_averaging`, `auto_ssa`

We use a 500-sample segment with a small window length to keep `auto_ssa`
(which builds a full pairwise d_corr matrix) tractable.

In [80]:
sig_ssa = signal[:500]
L = 50
X_hankel = build_trajectory_matrix(sig_ssa, L=L)
print("Hankel trajectory matrix shape:", X_hankel.shape)
assert X_hankel.shape == (L, len(sig_ssa) - L + 1)

U, S, Vt = svd_decompose(X_hankel, rank=5)
print("Top-5 singular values:", np.round(S, 3))
assert U.shape == (L, 5) and Vt.shape == (5, len(sig_ssa) - L + 1)

# reconstruct the rank-1 approximation via diagonal averaging
X1 = S[0] * np.outer(U[:, 0], Vt[0, :])
recon_rank1 = diagonal_averaging(X1)
print("Rank-1 reconstruction length:", len(recon_rank1))

# autoSSA hierarchical grouping
ssa_groups = auto_ssa(sig_ssa, r=3, L=L)
print(f"auto_ssa returned {len(ssa_groups)} grouped components, "
      f"each of length {len(ssa_groups[0])}")

# SSA engine wrapper
ssa_eng = get_engine("ssa", fs=FS, n_components=3, window_length=L)
ssa_eng_components = ssa_eng.fit(sig_ssa)
print(f"SSA engine produced {len(ssa_eng_components)} components")


Hankel trajectory matrix shape: (50, 451)
Top-5 singular values: [79.037 75.57  64.177 61.782 38.222]
Rank-1 reconstruction length: 500
auto_ssa returned 3 grouped components, each of length 500
SSA engine produced 3 components


### 2.4 `RankOneUpdater` — USSA stub

The rank-1 SVD updater (Brand 2003) is a stub scheduled for Week 11 of the
thesis timeline; calling `.update()` raises `NotImplementedError`. We show that
the class can be instantiated and that calling its update method correctly
raises.

In [81]:
U0, S0, Vt0 = svd_decompose(X_hankel, rank=3)
updater = RankOneUpdater(U0, S0, Vt0, refresh_every=1000)
print("RankOneUpdater instantiated:", updater.U.shape, updater.S.shape, updater.Vt.shape)
try:
    updater.update(0.1)
except NotImplementedError as exc:
    print("Stub correctly raises:", exc)


RankOneUpdater instantiated: (50, 3) (3,) (3, 451)
Stub correctly raises: USSA rank-1 update — implement in Week 11


## 3. Similarity metrics

Each metric is demonstrated on pairs of SSD components.

In [82]:
g1 = ssd_comps_only[0]
g2 = ssd_comps_only[1] if len(ssd_comps_only) > 1 else ssd_comps_only[0]

print(f"d_corr(g1, g1) = {d_corr(g1, g1):.6f}   # should be ~0")
print(f"d_corr(g1, g2) = {d_corr(g1, g2):.4f}")
print(f"d_freq(g1, g2) = {d_freq(g1, g2, fs=FS):.2f} Hz")
print(f"w_correlation(g1, g2, L=200) = {w_correlation(g1, g2, L=200):.4f}")

# subspace angle between two U-blocks from SSA
U_a, _, _ = svd_decompose(build_trajectory_matrix(g1[:500], L=50), rank=2)
U_b, _, _ = svd_decompose(build_trajectory_matrix(g2[:500], L=50), rank=2)
theta = subspace_angle(U_a, U_b)
print(f"subspace_angle(U_a, U_b) = {theta:.4f} rad  ({np.degrees(theta):.1f} deg)")


d_corr(g1, g1) = 0.000000   # should be ~0
d_corr(g1, g2) = 0.8818
d_freq(g1, g2) = 23.44 Hz
w_correlation(g1, g2, L=200) = 0.1140
subspace_angle(U_a, U_b) = 1.3754 rad  (78.8 deg)


## 4. Streaming pipeline

### 4.1 `WindowManager` — circular buffer + stride

We push samples one-by-one, confirm that windows are emitted exactly at stride
boundaries and that the `overlap` property equals `window_len - stride`.

In [83]:
WLEN, STRIDE = 300, 150
wm = WindowManager(window_len=WLEN, stride=STRIDE, fs=FS)
assert wm.overlap == WLEN - STRIDE == 150

emitted = []
for i, x in enumerate(signal):
    w = wm.push(float(x))
    if w is not None:
        emitted.append((i, w))
print(f"Emitted {len(emitted)} windows (first at sample {emitted[0][0]}, "
      f"last at sample {emitted[-1][0]})")
assert all(w.shape == (WLEN,) for _, w in emitted)
# stride check
deltas = np.diff([i for i, _ in emitted])
assert np.all(deltas == STRIDE)
print("stride spacing verified:", deltas[:5], "...")
wm.reset()


Emitted 19 windows (first at sample 299, last at sample 2999)
stride spacing verified: [150 150 150 150 150] ...


### 4.2 `ComponentMatcher` — stateless + stateful APIs, all distance modes

In [84]:
# prepare two consecutive windows and their SSD components
ssd_small = SSD(fs=FS, nmse_threshold=0.02, max_iter=5)
w_prev = signal[:WLEN]
w_curr = signal[STRIDE:STRIDE + WLEN]
comps_prev = ssd_small.fit(w_prev)[:-1]
comps_curr = ssd_small.fit(w_curr)[:-1]
print(f"prev window: {len(comps_prev)} comps  |  curr window: {len(comps_curr)} comps")

# --- stateless API ---
matcher_sl = ComponentMatcher(distance="d_corr", fs=FS)
cost_mat = matcher_sl.build_cost_matrix(comps_prev, comps_curr, overlap=WLEN - STRIDE)
stateless_mapping = matcher_sl.match(comps_prev, comps_curr, overlap=WLEN - STRIDE)
print("cost matrix shape:", cost_mat.shape)
print("stateless mapping {curr->prev}:", stateless_mapping)

# --- stateful API with persistent trajectory ids ---
matcher_sf = ComponentMatcher(distance="hybrid", freq_weight=0.5, fs=FS,
                              lookback=3, max_cost=0.6, max_trajectories=6)
m0 = matcher_sf.match_stateful(comps_prev, overlap=WLEN - STRIDE)
m1 = matcher_sf.match_stateful(comps_curr, overlap=WLEN - STRIDE)
prev_win_map = matcher_sf.previous_window_mapping()
print("stateful window 0:", m0)
print("stateful window 1:", m1)
print("previous_window_mapping:", prev_win_map)


prev window: 2 comps  |  curr window: 2 comps
cost matrix shape: (2, 2)
stateless mapping {curr->prev}: {0: 0, 1: 1}
stateful window 0: {0: 0, 1: 1}
stateful window 1: {0: 0, 1: 1}
previous_window_mapping: {0: 0, 1: 1}


In [85]:
# --- side-by-side comparison of all three distance modes ---
rows = []
for dist in ["d_corr", "d_freq", "hybrid"]:
    m = ComponentMatcher(distance=dist, freq_weight=0.5, fs=FS,
                         lookback=3, max_cost=0.9, max_trajectories=6)
    m.match_stateful(comps_prev, overlap=WLEN - STRIDE)
    mapping = m.match_stateful(comps_curr, overlap=WLEN - STRIDE)
    cost = m.build_cost_matrix(comps_prev, comps_curr, overlap=WLEN - STRIDE)
    prev_map = m.previous_window_mapping()
    mc = matching_confidence(cost, prev_map)
    n_traj = len(set(mapping.values()) - {-1})
    recon = np.sum(comps_curr, axis=0) if comps_curr else np.zeros(WLEN)
    rows.append({"distance": dist, "n_trajectories": n_traj,
                 "mean_confidence": round(mc, 3),
                 "qrf_db": round(qrf(w_curr, recon), 2)})
df_compare = pd.DataFrame(rows)
print(df_compare.to_string(index=False))


distance  n_trajectories  mean_confidence  qrf_db
  d_corr               2            0.601   18.33
  d_freq               2            1.000   18.33
  hybrid               2            0.784   18.33


### 4.3 `TrajectoryStore` — overlap averaging and `max_components` cap

In [86]:
store_demo = TrajectoryStore(max_components=4, max_len=len(signal))
# fake two overlapping windows to exercise the averaging logic
fake_comp_a = np.ones(WLEN)
fake_comp_b = 2 * np.ones(WLEN)
store_demo.update(0, [fake_comp_a], {0: None}, overlap=0)
store_demo.update(STRIDE, [fake_comp_b], {0: 0}, overlap=WLEN - STRIDE)
traj0 = store_demo.get(0)
print("trajectory 0 length:", len(traj0))
print("value in overlap region (should average 1 and 2 -> 1.5):",
      np.nanmean(traj0[STRIDE:WLEN]))

# max_components enforcement
small = TrajectoryStore(max_components=1, max_len=500)
small.update(0, [np.zeros(WLEN), np.ones(WLEN)], {0: None, 1: None}, overlap=0)
assert len(small.get_all()) == 1
print("max_components cap enforced:", list(small.get_all().keys()))


trajectory 0 length: 450
value in overlap region (should average 1 and 2 -> 1.5): 1.5
max_components cap enforced: [0]


### 4.4 Full streaming loop

One sample at a time -> `WindowManager` -> `SSD.fit` -> `ComponentMatcher.match_stateful`
-> `TrajectoryStore.update`. We also collect per-window metrics (QRF, singular
value drift, energy continuity, matching confidence, dominant frequency) so
that Section 5 can demonstrate every stability metric.

In [87]:
wm = WindowManager(window_len=WLEN, stride=STRIDE, fs=FS)
engine = SSD(fs=FS, nmse_threshold=0.01, max_iter=8)
matcher = ComponentMatcher(distance="hybrid", freq_weight=0.5, fs=FS,
                           lookback=10, max_cost=0.6, max_trajectories=8)
MAX_COMPONENTS = 8
store = TrajectoryStore(max_components=MAX_COMPONENTS, max_len=len(signal))

overlap = wm.overlap
prev_components = None
prev_S = None
metrics_rows = []
pipeline_records = []
window_idx = 0

for sample_idx, x in enumerate(signal):
    window = wm.push(float(x))
    if window is None:
        continue

    components = engine.fit(window)
    comps = components[:-1]

    mapping = dict(matcher.match_stateful(comps, overlap))
    prev_win_map = matcher.previous_window_mapping()

    window_start = sample_idx - WLEN + 1
    store.update(window_start, comps, mapping, overlap)

    recon = np.sum(comps, axis=0) if comps else np.zeros_like(window)
    q_val = qrf(window, recon)

    # engine-agnostic SVD drift using a Hankel embedding of the raw window
    Lw = max(2, len(window) // 3)
    Xw = build_trajectory_matrix(window, Lw)
    _, S_curr, _ = svd_decompose(Xw)
    sv_drift = singular_value_drift(S_curr, prev_S)
    prev_S = S_curr

    ec = energy_continuity(comps, prev_components, prev_win_map)

    if prev_components is not None and prev_win_map:
        C = matcher.build_cost_matrix(prev_components, comps, overlap)
        mc = matching_confidence(C, prev_win_map)
    else:
        mc = float("nan")

    row = {"window_index": window_idx, "qrf": q_val,
           "singular_value_drift": sv_drift,
           "energy_continuity": ec, "matching_confidence": mc}
    for ci, comp in enumerate(comps):
        row[f"f_max_c{ci}"] = dominant_frequency(comp, fs=FS)
    for ci in range(len(comps), MAX_COMPONENTS):
        row[f"f_max_c{ci}"] = np.nan
    metrics_rows.append(row)

    pipeline_records.append({
        "window_idx": window_idx, "sample_start": window_start,
        "window_signal": window.copy(), "components": [c.copy() for c in comps],
    })

    prev_components = comps
    window_idx += 1

metrics_df = pd.DataFrame(metrics_rows)
metrics_csv_path = RESULTS_DIR / "metrics.csv"
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"Processed {window_idx} windows. Metrics saved to {metrics_csv_path}")
print(metrics_df.head().to_string())


Processed 19 windows. Metrics saved to ../results/demo_run/metrics.csv
   window_index        qrf  singular_value_drift  energy_continuity  matching_confidence  f_max_c0  f_max_c1  f_max_c2   f_max_c3   f_max_c4  f_max_c5  f_max_c6  f_max_c7
0             0  20.932690                   NaN                NaN                  NaN  50.78125  15.62500  19.53125   50.78125        NaN       NaN       NaN       NaN
1             1  20.908202              3.747692         306.758555             0.800660  23.43750  50.78125  50.78125        NaN        NaN       NaN       NaN       NaN
2             2  22.890389              3.784173         696.632681             0.820131  31.25000  50.78125  31.25000  148.43750        NaN       NaN       NaN       NaN
3             3  25.235312              5.683492        2047.258221             0.857793  50.78125  35.15625   0.00000   50.78125  285.15625       NaN       NaN       NaN
4             4  20.464529             39.196116        3485.248975       

## 5. Stability metrics — summary after the streaming run

Every metric from `src/metrics/stability.py` has been computed inside the
streaming loop (Section 4.4). Here we aggregate and display them, including
the **global** `freq_drift_aggregate` which is a post-hoc statistic.

In [88]:
summary = {}
summary["mean_qrf_db"]             = float(metrics_df["qrf"].replace([np.inf], np.nan).mean())
summary["mean_nmse_last_window"]   = float(nmse(pipeline_records[-1]["window_signal"]
                                                 - np.sum(pipeline_records[-1]["components"], axis=0),
                                                 pipeline_records[-1]["window_signal"]))
summary["mean_energy_continuity"]  = float(metrics_df["energy_continuity"].mean(skipna=True))
summary["mean_singular_value_drift"] = float(metrics_df["singular_value_drift"].mean(skipna=True))
summary["mean_matching_confidence"] = float(metrics_df["matching_confidence"].mean(skipna=True))

fmax_cols = sorted([c for c in metrics_df.columns if c.startswith("f_max_c")])
freq_drift_per_component = {c: freq_drift_aggregate(metrics_df[c].values) for c in fmax_cols}

for k, v in summary.items():
    print(f"{k:35s} = {v:.4f}")
print("\nfreq_drift_aggregate per component (Var_t[f_max]):")
for c, v in freq_drift_per_component.items():
    print(f"  {c}: {v:.4f}" if np.isfinite(v) else f"  {c}: NaN")

run_summary_path = RESULTS_DIR / "run_summary.json"
with open(run_summary_path, "w") as fh:
    json.dump({**summary,
               **{f"freq_drift_c{i}": (None if not np.isfinite(v) else v)
                  for i, (_, v) in enumerate(freq_drift_per_component.items())}},
              fh, indent=2)
print("\nSaved:", run_summary_path)


mean_qrf_db                         = 22.2811
mean_nmse_last_window               = 0.0063
mean_energy_continuity              = 2883.4774
mean_singular_value_drift           = 10.5088
mean_matching_confidence            = 0.8949

freq_drift_aggregate per component (Var_t[f_max]):
  f_max_c0: 58.4145
  f_max_c1: 1400.3239
  f_max_c2: 3402.7100
  f_max_c3: 11351.5854
  f_max_c4: NaN
  f_max_c5: NaN
  f_max_c6: NaN
  f_max_c7: NaN

Saved: ../results/demo_run/run_summary.json


## 6. Visualizations — every plotting function in `src/visualization/`

In [89]:
# 6.1 plot_decomposition — offline SSD component breakdown
plot_decomposition(signal, ssd_comps_only, residual=ssd_residual, fs=FS,
                   title="SSD Decomposition (offline)",
                   save_path=str(RESULTS_DIR / "06_decomposition.png"))

# 6.2 plot_trajectory_overlay
plot_trajectory_overlay(store, signal, fs=FS,
                        save_path=str(RESULTS_DIR / "06_trajectory_overlay.png"))

# 6.3 plot_component_spectra
plot_component_spectra(ssd_comps_only, fs=FS, nperseg=256,
                       save_path=str(RESULTS_DIR / "06_component_spectra.png"))

# 6.4 plot_metrics_over_windows (reads CSV)
plot_metrics_over_windows(str(metrics_csv_path),
                          save_path=str(RESULTS_DIR / "06_metrics_over_windows.png"))
print("Saved 6.1-6.4")


Saved 6.1-6.4


In [90]:
# 6.5 plot_pipeline_dashboard
plot_pipeline_dashboard(signal, ssd_comps_only, store, str(metrics_csv_path), fs=FS,
                        save_path=str(RESULTS_DIR / "06_pipeline_dashboard.png"))

# 6.6 plot_window_reconstruction for first / middle / last window
n_rec = len(pipeline_records)
for label, idx in [("first", 0), ("middle", n_rec // 2), ("last", n_rec - 1)]:
    rec = pipeline_records[idx]
    plot_window_reconstruction(rec["window_signal"], rec["components"],
                               rec["window_idx"], rec["sample_start"], fs=FS,
                               save_path=str(RESULTS_DIR / f"06_window_{label}.png"))

# 6.7 plot_window_grid (9-window mosaic)
plot_window_grid(pipeline_records, n_windows=9, fs=FS,
                 save_path=str(RESULTS_DIR / "06_window_grid.png"))
print("Saved 6.5-6.7")


Saved 6.5-6.7


In [91]:
# 6.8 plot_nmse_over_time from the trajectory store
trajs = store.get_all()
full_recon = np.zeros_like(signal, dtype=np.float64)
coverage = np.zeros(len(signal), dtype=np.float64)
for arr in trajs.values():
    k = min(len(arr), len(full_recon))
    valid = ~np.isnan(arr[:k])
    full_recon[:k][valid] += arr[:k][valid]
    coverage[:k][valid] += 1.0
# mark positions with zero coverage as NaN
full_recon_plot = full_recon.copy()
full_recon_plot[coverage == 0] = np.nan

plot_nmse_over_time(signal, full_recon_plot, fs=FS,
                    save_path=str(RESULTS_DIR / "06_nmse_over_time.png"))

# 6.9 plot_matching_graph between two consecutive windows (stateless API)
ms = ComponentMatcher(distance="hybrid", freq_weight=0.5, fs=FS)
C_vis = ms.build_cost_matrix(comps_prev, comps_curr, overlap=WLEN - STRIDE)
map_vis = ms.match(comps_prev, comps_curr, overlap=WLEN - STRIDE)
plot_matching_graph(comps_prev, comps_curr, map_vis,
                    overlap=WLEN - STRIDE, cost_matrix=C_vis,
                    save_path=str(RESULTS_DIR / "06_matching_graph.png"))

# 6.10 plot_metrics (advanced 3-panel: SV drift, freq trajectories, freq_drift bar)
plot_metrics(RESULTS_DIR, show=False)
print("Saved 6.8-6.10")


Saved 6.8-6.10


## 7. Robustness: QRF vs SNR sweep

We decompose the primary signal under increasing noise and tabulate the
quality of reconstruction (QRF) and normalised MSE. This confirms graceful
degradation at low SNR.

In [92]:
snr_rows = []
for snr in [5, 10, 15, 20, 30]:
    noisy = chirp_plus_sinusoid(N=N, f_sin=50.0, f_start=10.0, f_end=150.0,
                                fs=FS, snr_db=snr, seed=SEED)
    comps = SSD(fs=FS, nmse_threshold=0.01, max_iter=8).fit(noisy)
    recon = np.sum(comps[:-1], axis=0)
    snr_rows.append({"snr_db": snr,
                     "qrf_db": round(qrf(noisy, recon), 2),
                     "nmse": round(nmse(noisy - recon, noisy), 5),
                     "n_components": len(comps) - 1})
snr_df = pd.DataFrame(snr_rows)
snr_df.to_csv(RESULTS_DIR / "07_snr_sweep.csv", index=False)
print(snr_df.to_string(index=False))


 snr_db  qrf_db    nmse  n_components
      5   22.45 0.00569             6
     10   17.61 0.01734             8
     15   18.92 0.01282             8
     20   22.00 0.00630             5
     30   27.69 0.00170             5


## 8. Dynamic component onset detection

Using `component_onset`, we run the streaming pipeline and track the number
of active trajectories over time. The `max_trajectories` cap and reusable-id
allocation inside `TrajectoryStore._next_free_id` prevent trajectory-id
inflation.

In [93]:
onset_sig = component_onset(N=4000, f_steady=30.0, f_onset=120.0,
                            onset_sample=2000, fs=FS, seed=SEED)

wm2 = WindowManager(window_len=400, stride=200, fs=FS)
eng2 = SSD(fs=FS, nmse_threshold=0.02, max_iter=6)
match2 = ComponentMatcher(distance="hybrid", freq_weight=0.5, fs=FS,
                          lookback=10, max_cost=0.6, max_trajectories=4)
store2 = TrajectoryStore(max_components=4, max_len=len(onset_sig))

active_counts = []
for i, x in enumerate(onset_sig):
    w = wm2.push(float(x))
    if w is None:
        continue
    comps = eng2.fit(w)[:-1]
    mapping = dict(match2.match_stateful(comps, wm2.overlap))
    store2.update(i - wm2.window_len + 1, comps, mapping, wm2.overlap)
    active_counts.append((i / FS, len(set(mapping.values()) - {-1})))

tt, cc = zip(*active_counts)
fig, ax = plt.subplots(figsize=(10, 3))
ax.step(tt, cc, where="post", color="purple")
ax.axvline(2000 / FS, color="red", linestyle="--", label="onset at t=2.0 s")
ax.set_xlabel("time (s)"); ax.set_ylabel("# active components")
ax.set_title("Component onset detection")
ax.legend(); fig.tight_layout()
fig.savefig(RESULTS_DIR / "08_onset_detection.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Trajectories stored: {len(store2.get_all())} (capped by max_components=4)")


Trajectories stored: 2 (capped by max_components=4)


## 9. Experiment runner integration

`experiments/run_experiment.py` ties everything together via a YAML config.
We build the pipeline from a config dict using `build_pipeline`, then execute
a full run via `run_experiment(config_dict=...)`, inspecting the exported
`metrics.csv` and `run_summary.json`.

In [94]:
with open("../experiments/configs/baseline.yaml") as fh:
    baseline_cfg = yaml.safe_load(fh)
print("baseline.yaml:")
print(yaml.dump(baseline_cfg, sort_keys=False))

# build_pipeline demonstration (without running the full loop)
cfg_copy = yaml.safe_load(yaml.dump(baseline_cfg))
wm_bp, eng_bp, match_bp, store_bp = build_pipeline(cfg_copy,
                                                   signal_length=baseline_cfg["signal"]["N"])
print("build_pipeline produced:",
      type(wm_bp).__name__, type(eng_bp).__name__,
      type(match_bp).__name__, type(store_bp).__name__)


baseline.yaml:
signal:
  type: chirp_plus_sinusoid
  N: 3000
  fs: 1000.0
  f_sin: 50.0
  f_start: 10.0
  f_end: 150.0
  snr_db: 20.0
streaming:
  window_len: 300
  stride: 150
  max_components: 100
engine:
  name: ssd
  nmse_threshold: 0.01
  max_iter: 20
matcher:
  distance: hybrid
  freq_weight: 0.5
  lookback: 10
  max_cost: 0.6
output:
  save_trajectories: true
  save_metrics: true

build_pipeline produced: WindowManager SSD ComponentMatcher TrajectoryStore


In [95]:
# Full end-to-end experiment via run_experiment(config_dict=...)
exp_cfg = yaml.safe_load(yaml.dump(baseline_cfg))
exp_out = RESULTS_DIR / "baseline_run"
run_experiment(config_dict=exp_cfg, output_dir=str(exp_out))

exp_metrics = pd.read_csv(exp_out / "metrics.csv")
with open(exp_out / "run_summary.json") as fh:
    exp_summary = json.load(fh)

print(f"metrics.csv: {exp_metrics.shape[0]} windows, "
      f"columns = {list(exp_metrics.columns)[:6]}...")
print("run_summary.json freq_drift entries:",
      {k: v for k, v in exp_summary.items() if k.startswith("freq_drift_")})
print(exp_metrics.head().to_string())


Streaming: 100%|██████████| 3000/3000 [00:01<00:00, 1946.17it/s]

metrics.csv: 19 windows, columns = ['window_index', 'qrf', 'singular_value_drift', 'energy_continuity', 'matching_confidence', 'f_max_c0']...
run_summary.json freq_drift entries: {'freq_drift_c0': 58.4145331977147, 'freq_drift_c1': 1400.3238875973184, 'freq_drift_c2': 3402.7099609375, 'freq_drift_c3': 11351.585388183594, 'freq_drift_c4': None, 'freq_drift_c5': None, 'freq_drift_c6': None, 'freq_drift_c7': None, 'freq_drift_c8': None, 'freq_drift_c9': None, 'freq_drift_c10': None, 'freq_drift_c11': None, 'freq_drift_c12': None, 'freq_drift_c13': None, 'freq_drift_c14': None, 'freq_drift_c15': None, 'freq_drift_c16': None, 'freq_drift_c17': None, 'freq_drift_c18': None, 'freq_drift_c19': None, 'freq_drift_c20': None, 'freq_drift_c21': None, 'freq_drift_c22': None, 'freq_drift_c23': None, 'freq_drift_c24': None, 'freq_drift_c25': None, 'freq_drift_c26': None, 'freq_drift_c27': None, 'freq_drift_c28': None, 'freq_drift_c29': None, 'freq_drift_c30': None, 'freq_drift_c31': None, 'freq_drift

## 10. Coverage summary

| Module | Symbol | Section |
|---|---|---|
| `experiments/synthetic/generators.py` | `two_sinusoids`, `chirp_plus_sinusoid`, `rossler`, `component_onset` | 1, 8 |
| `src/engines/base.py` | `DecompositionEngine` | 2.1 |
| `src/engines/__init__.py` | `get_engine` | 2.1 |
| `src/engines/ssd.py` | `SSD.fit`, `_build_trajectory_matrix`, `_choose_window_length` | 2.2 |
| `src/engines/ssa.py` | `build_trajectory_matrix`, `svd_decompose`, `diagonal_averaging`, `auto_ssa`, `SSA` | 2.3 |
| `src/engines/svd_update.py` | `RankOneUpdater` (stub) | 2.4 |
| `src/metrics/similarity.py` | `d_corr`, `d_freq`, `w_correlation`, `subspace_angle` | 3 |
| `src/streaming/window_manager.py` | `WindowManager` (`push`, `overlap`, `reset`) | 4.1 |
| `src/streaming/component_matcher.py` | `ComponentMatcher.match`, `build_cost_matrix`, `match_stateful`, `previous_window_mapping`, all 3 distances, `max_cost`, `max_trajectories`, `lookback` | 4.2 |
| `src/streaming/trajectory_store.py` | `TrajectoryStore.update`, `get`, `get_all`, `max_components` | 4.3 |
| (full loop) | streaming pipeline end-to-end | 4.4 |
| `src/metrics/stability.py` | `qrf`, `nmse`, `energy_continuity`, `singular_value_drift`, `dominant_frequency`, `freq_drift_aggregate`, `matching_confidence` | 4.4, 5 |
| `src/visualization/component_plots.py` | `plot_decomposition`, `plot_trajectory_overlay`, `plot_component_spectra`, `plot_matching_graph` | 6 |
| `src/visualization/metrics_plots.py` | `plot_metrics_over_windows` | 6 |
| `src/visualization/pipeline_dashboard.py` | `plot_pipeline_dashboard` | 6 |
| `src/visualization/window_inspector.py` | `plot_window_reconstruction`, `plot_window_grid`, `plot_nmse_over_time` | 6 |
| `src/visualization/plot_metrics.py` | `plot_metrics` | 6 |
| (robustness) | QRF / NMSE vs SNR sweep | 7 |
| (dynamics) | component onset detection + reusable-id resolver | 8 |
| `experiments/run_experiment.py` | `build_pipeline`, `run` | 9 |

All framework features have been demonstrated. End of notebook.
